# KAIROS: NIFTY 50 Market Movement Analysis
UML501 — Machine Learning

Single-model research notebook: NIFTY 50 and India VIX, seven features, one balanced Random Forest.

**Execution:** Place the two source CSV files in the working directory or set `KAIROS_DATA_DIR` to their location. The optional acquisition function is not executed by default.

**Development assistance:** AI-assisted coding and editing were used; the methodology and results should be independently reviewed.

## 1. Problem statement
Investigate whether historical NIFTY 50 prices and India VIX can predict significant net price movements over the next five trading sessions. Forecasts are issued after market close. The provisional threshold is ±3%.

## 2. Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import os

## 3. Data collection
### 3.1 Source and collection method
Historical NIFTY 50 observations were obtained from NSE Indices. The optional acquisition function below reconstructs the original batched HTTP request procedure; the live endpoint has not been retested. Fixed CSV snapshots are used for reproducibility. The original India VIX acquisition script is unavailable.
### 3.2 Optional NIFTY acquisition code

In [ ]:
def collect_nifty_history(start_date, end_date):
    import requests
    import json
    import time
    from datetime import timedelta

    url = 'https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString'
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Content-Type': 'application/json',
        'X-Requested-With': 'XMLHttpRequest',
        'Referer': 'https://www.niftyindices.com/reports/historical-data',
        'Origin': 'https://www.niftyindices.com'
    }
    if start_date > end_date:
        raise ValueError('Start date must precede end date.')
    frames = []
    with requests.Session() as session:
        session.headers.update(headers)
        current_start = start_date
        while current_start <= end_date:
            current_end = min(current_start + timedelta(days=364), end_date)
            cinfo = ("{'name':'NIFTY 50','startDate':'" + current_start.strftime('%m/%d/%Y')
                     + "','endDate':'" + current_end.strftime('%m/%d/%Y')
                     + "','indexName':'NIFTY 50'}")
            for attempt in range(3):
                try:
                    response = session.post(url, json={'cinfo': cinfo}, timeout=90)
                    response.raise_for_status()
                    result = response.json()
                    if isinstance(result, dict):
                        result = result['d']
                    if isinstance(result, str):
                        result = json.loads(result)
                    if not isinstance(result, list):
                        raise ValueError('Unexpected response format')
                    if result:
                        frames.append(pd.DataFrame(result))
                    elif (current_end - current_start).days >= 7:
                        raise ValueError('Unexpected empty multi-day batch')
                    break
                except (requests.RequestException, ValueError, KeyError) as exc:
                    if attempt == 2:
                        raise RuntimeError(f'Failed batch {current_start} to {current_end}') from exc
                    time.sleep(2 ** attempt)
            current_start = current_end + timedelta(days=1)
            time.sleep(1.5)
    if not frames:
        raise ValueError('No observations returned')
    result = pd.concat(frames, ignore_index=True).rename(columns={
        'HistoricalDate': 'Date', 'date': 'Date', 'OPEN': 'Open',
        'HIGH': 'High', 'LOW': 'Low', 'CLOSE': 'Close'
    })[['Date', 'Open', 'High', 'Low', 'Close']]
    result['Date'] = pd.to_datetime(result['Date'], errors='raise')
    for col in ['Open', 'High', 'Low', 'Close']:
        result[col] = pd.to_numeric(result[col], errors='raise')
    return result.sort_values('Date').reset_index(drop=True)


### 3.3 Load source datasets
The source CSV files remain unchanged throughout the analysis.

In [ ]:
base = Path(os.environ.get('KAIROS_DATA_DIR', str(Path.cwd())))
nifty_file = base / 'KAIROS_NIFTY50_NSE_2006_2026.csv'
vix_file = base / 'KAIROS_INDIA_VIX_2008_2026.csv'
for file in (nifty_file, vix_file):
    if not file.is_file():
        raise FileNotFoundError(f'Upload {file.name} to {base} before continuing.')
nifty = pd.read_csv(nifty_file)
vix = pd.read_csv(vix_file)
print('NIFTY:', nifty.shape)
print('VIX:', vix.shape)
display(nifty.head(), vix.head())

## 4. Data preprocessing
### 4.1 Check column names and data types

In [ ]:
prices = ['Open', 'High', 'Low', 'Close']
assert set(nifty.columns) == set(['Date'] + prices)
assert set(vix.columns) == {'Date', 'VIX_Close'}
print(nifty.dtypes)
print(vix.dtypes)

### 4.2 Convert dates and sort chronologically

In [ ]:
for frame in (nifty, vix):
    frame['Date'] = pd.to_datetime(frame['Date'], errors='raise')
    frame.sort_values('Date', inplace=True)
    frame.reset_index(drop=True, inplace=True)
    assert frame.Date.notna().all()
    assert frame.Date.is_monotonic_increasing
print('NIFTY:', nifty.Date.min().date(), 'to', nifty.Date.max().date())
print('VIX:', vix.Date.min().date(), 'to', vix.Date.max().date())

### 4.3 Check missing values

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'missing values:')
    print(frame.isna().sum())
    assert frame.notna().all().all()

### 4.4 Check duplicate dates

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'duplicate dates:', frame.Date.duplicated().sum())
    assert frame.Date.is_unique

### 4.5 Validate OHLC and VIX values

In [ ]:
assert np.isfinite(nifty[prices]).all().all()
assert (nifty[prices] > 0).all().all()
assert (nifty.High >= nifty[['Open', 'Close', 'Low']].max(axis=1)).all()
assert (nifty.Low <= nifty[['Open', 'Close', 'High']].min(axis=1)).all()
assert np.isfinite(vix.VIX_Close).all() and (vix.VIX_Close > 0).all()
print('OHLC and VIX checks passed.')

### 4.6 Align the date ranges
Use the common observation period. Earlier NIFTY records are excluded from the joint experiment.

In [ ]:
start = max(nifty.Date.min(), vix.Date.min())
end = min(nifty.Date.max(), vix.Date.max())
assert start <= end
aligned = nifty.loc[nifty.Date.between(start, end)].copy()
vix_aligned = vix.loc[vix.Date.between(start, end)].copy()
assert vix_aligned.Date.isin(aligned.Date).all()
print('Common period:', start.date(), 'to', end.date())
print('Excluded NIFTY rows:', len(nifty) - len(aligned))
print('Aligned NIFTY rows:', len(aligned))
print('Aligned VIX rows:', len(vix_aligned))

### 4.7 Merge the datasets
Use a one-to-one left join to preserve aligned NIFTY sessions.

In [ ]:
df = aligned.merge(vix_aligned, on='Date', how='left', validate='one_to_one')
assert len(df) == len(aligned)
assert df.Date.is_unique and df.Date.is_monotonic_increasing
print('Merged shape before missing-value flag:', df.shape)
print('Missing VIX observations:', df.VIX_Close.isna().sum())
display(df.loc[df.VIX_Close.isna()])

### 4.8 Handle missing VIX observations
Flag missing readings before carrying forward the last available earlier value. A carried-forward reading is not an observed value.

In [ ]:
df['VIX_missing'] = df.VIX_Close.isna().astype(int)
print('Missing VIX dates:', df.loc[df.VIX_missing.eq(1), 'Date'].dt.date.tolist())
df['VIX_Close'] = df.VIX_Close.ffill()
assert df.notna().all().all()
assert len(df) == len(aligned)
print('Remaining missing values:', df.isna().sum().sum())
print('Cleaned dataset shape:', df.shape)
display(df.head())

### 4.9 Export the cleaned combined dataset
The market-data export excludes the internal imputation flag. A separate audit file records the affected dates and carried-forward VIX values. The internal flag is retained for feature-quality checks.

In [ ]:
output_path = base / 'KAIROS_COMBINED_CLEANED.csv'
audit_path = base / 'KAIROS_VIX_IMPUTATION_AUDIT.csv'
combined = df.drop(columns=['VIX_missing'])
combined.to_csv(output_path, index=False)
df.loc[df.VIX_missing.eq(1), ['Date', 'VIX_Close']].to_csv(audit_path, index=False)
print('Combined dataset:', combined.shape)
print('Imputation audit:', audit_path.name)

## 5. Chronological experiment boundaries
Use 70% for training, 15% for validation and 15% for testing. EDA uses the training period only. The test partition is a procedural holdout rather than an independent external dataset because the historical snapshot was previously examined.

In [ ]:
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)
train = df.iloc[:train_end].copy()
for name, part in [('Train', train), ('Validation', df.iloc[train_end:val_end]), ('Test', df.iloc[val_end:])]:
    print(name, len(part), part.Date.min().date(), part.Date.max().date())

## 6. Exploratory Data Analysis (EDA)
### 6.1 Descriptive statistics

In [ ]:
display(train[prices + ['VIX_Close']].describe().round(2))

### 6.2 Market trends
NIFTY closing levels and implied volatility are plotted separately.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Close, linewidth=1)
axes[0].set(title='NIFTY 50 closing level', ylabel='Index points')
axes[1].plot(train.Date, train.VIX_Close, linewidth=1)
axes[1].set(title='India VIX', ylabel='Index level', xlabel='Date')
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### 6.3 Returns and rolling volatility
Compute daily percentage returns and trailing 20-session standard deviation.

In [ ]:
train['Return_1d'] = train.Close.pct_change() * 100
train['Volatility_20d'] = train.Return_1d.rolling(20).std()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Return_1d, linewidth=0.7)
axes[0].set(title='Daily NIFTY returns', ylabel='Return (%)')
axes[1].plot(train.Date, train.Volatility_20d, linewidth=1)
axes[1].set(title='20-session rolling volatility', ylabel='Daily std. dev. (%)', xlabel='Date')
fig.tight_layout()
plt.show()
print(train.Return_1d.describe().round(3))
print('Largest daily decline:', train.loc[train.Return_1d.idxmin(), ['Date','Return_1d']].to_dict())
print('Largest daily rise:', train.loc[train.Return_1d.idxmax(), ['Date','Return_1d']].to_dict())

### 6.4 VIX and market returns
Pearson correlation measures linear association, not predictive performance.

In [ ]:
train['VIX_Change'] = train.VIX_Close.pct_change() * 100
train['Abs_Return'] = train.Return_1d.abs()
cols = ['Return_1d', 'Abs_Return', 'VIX_Close', 'VIX_Change', 'Volatility_20d']
corr = train[cols].corr()
display(corr.round(3))
plt.figure(figsize=(7, 4))
plt.scatter(train.VIX_Change, train.Return_1d, s=8, alpha=0.35)
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.xlabel('Daily VIX change (%)')
plt.ylabel('Daily NIFTY return (%)')
plt.title('Same-session return and VIX change')
plt.tight_layout()
plt.show()

### 6.5 Candidate target distribution
Use a provisional five-session return threshold of ±3%. Exclude unknown outcomes.

In [ ]:
forward = (train.Close.shift(-5) / train.Close - 1) * 100
observed = forward.dropna()
boundary = np.isclose(observed.abs(), 3, rtol=0, atol=1e-10)
labels = pd.Series(np.select([(observed < -3) & ~boundary, (observed > 3) & ~boundary], ['Major Down', 'Major Up'], default='Ordinary'), index=observed.index)
order = ['Major Down', 'Ordinary', 'Major Up']
counts = labels.value_counts().reindex(order, fill_value=0)
summary = pd.DataFrame({'Count': counts, 'Percent': counts / len(labels) * 100})
display(summary.round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(observed, bins=60, edgecolor='white')
for boundary in (-3, 3):
    axes[0].axvline(boundary, linestyle='--', linewidth=1.4)
axes[0].set(title='Five-session forward returns', xlabel='Return (%)', ylabel='Observations')
axes[1].bar(order, counts)
axes[1].set(title='Candidate target distribution', ylabel='Observations')
fig.tight_layout()
plt.show()
assert len(labels) == len(train) - 5
print('Excluded rows without a within-training outcome:', len(train) - len(labels))

### 6.6 EDA conclusions
Class imbalance requires macro F1 and per-class precision and recall alongside accuracy. Same-session correlations are not evidence of future forecasting skill.

## 7. Feature engineering
### 7.1 Candidate predictors
Forecasts are issued after market close. Trailing features may use earlier history across partition boundaries. The imputation flag is retained for data-quality checks only, not as a predictor. The first 20 sessions are warm-up rows.

In [ ]:
def make_features(data):
    features = pd.DataFrame(index=data.index)
    close = data['Close']
    vix_close = data['VIX_Close']
    daily_return = close.pct_change(fill_method=None) * 100
    features['Return_1d'] = daily_return
    features['Return_5d'] = close.pct_change(5, fill_method=None) * 100
    features['Volatility_20d'] = daily_return.rolling(20).std()
    features['MA_20_gap_pct'] = (close / close.rolling(20).mean() - 1) * 100
    features['Intraday_range_pct'] = (data['High'] - data['Low']) / data['Open'] * 100
    features['VIX_Close'] = vix_close
    features['VIX_change_1d'] = vix_close.pct_change(fill_method=None) * 100
    unreliable = data['VIX_missing'].eq(1) | data['VIX_missing'].shift(1, fill_value=0).eq(1)
    features.loc[unreliable, 'VIX_change_1d'] = np.nan
    return features

X = make_features(df)
assert X.shape[1] == 7 and "VIX_missing" not in X.columns
X_train = X.iloc[:train_end].copy()
print('Candidate predictors:', X_train.shape[1])
print('Training rows with all predictors available:', X_train.dropna().shape[0])
display(X_train.iloc[20:25].round(3))

### 7.2 Feature validation and leakage checks
Prefix invariance tests for accidental use of future observations.

In [ ]:
for cutoff in (50, 250, train_end, val_end):
    pd.testing.assert_frame_equal(make_features(df.iloc[:cutoff]), X.iloc[:cutoff])
assert not np.isinf(X.to_numpy(dtype=float)).any()
assert X.index.equals(df.index)
print('Prefix-invariance, finite-value and alignment checks passed.')
print('Training missing counts:')
print(X_train.isna().sum())

## 8. Target variable creation
### 8.1 Five-session classification labels
Returns above +3% map to 1, below −3% to −1, and the inclusive interval to 0. A numerical tolerance protects threshold boundaries. Outcomes are generated separately in each partition.

In [ ]:
"""Five-session classification labels, with partition boundary protection."""
import numpy as np
import pandas as pd


def make_targets(data, horizon=5, threshold=3.0):
    if horizon < 1 or threshold <= 0:
        raise ValueError('horizon and threshold must be positive')
    if not data.Date.is_monotonic_increasing or not data.Date.is_unique:
        raise ValueError('Dates must be unique and sorted')
    forward = (data.Close.shift(-horizon) / data.Close - 1) * 100
    boundary = np.isclose(forward.abs(), threshold, rtol=0, atol=1e-10)
    target = pd.Series(pd.NA, index=data.index, dtype='Int64')
    target.loc[forward.notna()] = 0
    target.loc[(forward > threshold) & ~boundary] = 1
    target.loc[(forward < -threshold) & ~boundary] = -1
    return pd.DataFrame({'Outcome_date': data.Date.shift(-horizon),
                         'Forward_return_5d_pct': forward, 'Target': target})


### 8.2 Prepare aligned training observations
Forward returns and outcome dates are retained for evaluation only. Exclude unavailable features and unknown outcomes.

In [ ]:
partitions = {'train': slice(0, train_end), 'validation': slice(train_end, val_end), 'test': slice(val_end, len(df))}
feature_columns = X.columns.tolist()
training_labels = make_targets(df.iloc[partitions['train']])
training_mask = X.iloc[partitions['train']].notna().all(axis=1) & training_labels.Target.notna()
X_fit = X.iloc[partitions['train']].loc[training_mask, feature_columns].copy()
y_fit = training_labels.loc[training_mask, 'Target'].astype(int)
print('Eligible training observations:', len(y_fit))
print('Excluded for unknown outcomes:', training_labels.Target.isna().sum())
print('Excluded for unavailable predictors among labelled rows:', ((~X.iloc[partitions['train']].notna().all(axis=1)) & training_labels.Target.notna()).sum())
counts_fit = y_fit.value_counts().reindex([-1, 0, 1], fill_value=0)
print(pd.DataFrame({'Count': counts_fit, 'Percent': (counts_fit / len(y_fit) * 100).round(2)}))

### 8.3 Validate target boundaries and alignment

In [ ]:
for future, expected in [(96, -1), (97, 0), (100, 0), (103, 0), (104, 1)]:
    example = pd.DataFrame({'Date': pd.date_range('2020-01-01', periods=6), 'Close': [100]*5 + [future]})
    check = make_targets(example)
    assert check.Target.iloc[0] == expected
    assert check.Target.iloc[1:].isna().all()
for part in partitions.values():
    block = df.iloc[part]
    result = make_targets(block)
    assert result.Target.tail(5).isna().all()
    assert result.Outcome_date.dropna().le(block.Date.max()).all()
    assert result.Target.notna().sum() == len(block) - 5
assert X_fit.index.equals(y_fit.index)
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(X_fit.columns)
assert X_fit.notna().all().all()
print('Target threshold, partition boundary and feature alignment checks passed.')

## 9. Final dataset preparation
### 9.1 Training feature review
Review training-period missingness, variability and pairwise correlations. Correlation screening is descriptive; feature selection is deferred to chronological model validation.

In [ ]:
review = pd.DataFrame({'Missing': X_train.isna().sum(),
                       'Unique_values': X_train.nunique(),
                       'Std_dev': X_train.std()})
display(review.round(4))
assert X_train.nunique().gt(1).all()
correlations = X_train.corr()
pairs = correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1)).stack()
strong_pairs = pairs[pairs.abs() >= 0.80].sort_values(key=abs, ascending=False)
print('Feature correlations with |r| >= 0.80:')
print(strong_pairs.round(3) if len(strong_pairs) else 'None')

### 9.2 Assemble partition-safe datasets
Only observations with all seven features and a known outcome within their chronological partition are eligible. The research table contains market values, features, outcome metadata and partition identifiers. Only the explicit feature allowlist is used as model input.

In [ ]:
"""Assemble partition-safe research datasets from aligned prices and features."""
import pandas as pd
import numpy as np

FEATURE_COLUMNS = ['Return_1d', 'Return_5d', 'Volatility_20d', 'MA_20_gap_pct',
                   'Intraday_range_pct', 'VIX_Close', 'VIX_change_1d']
MARKET_COLUMNS = ['Date', 'Open', 'High', 'Low', 'Close', 'VIX_Close']


def build_datasets(df, X, partitions, make_targets):
    assert df.index.is_unique and X.index.equals(df.index)
    assert df.Date.is_unique and df.Date.is_monotonic_increasing
    assert X.columns.tolist() == FEATURE_COLUMNS
    pd.testing.assert_series_equal(X.VIX_Close, df.VIX_Close)
    assert not np.isinf(X.to_numpy(dtype=float)).any()
    assert list(partitions) == ['train', 'validation', 'test']
    used = []
    datasets = {}
    audit = []
    for name, section in partitions.items():
        market = df.iloc[section]
        features = X.iloc[section]
        used.extend(market.index.tolist())
        labels = make_targets(market)
        assert labels.index.equals(market.index)
        feature_ok = features.notna().all(axis=1)
        target_ok = labels.Target.notna()
        eligible = feature_ok & target_ok
        joined = market[MARKET_COLUMNS].join(features.drop(columns='VIX_Close')).join(labels)
        joined['Partition'] = name
        part = joined.loc[eligible].copy()
        part['Target'] = part.Target.astype(int)
        assert not part.empty and part.notna().all().all()
        assert part.Target.isin([-1, 0, 1]).all()
        assert part.Outcome_date.gt(part.Date).all()
        assert part.Outcome_date.le(market.Date.max()).all()
        assert part.Date.is_unique and part.Date.is_monotonic_increasing
        datasets[name] = part
        audit.append({'Partition': name, 'Original': len(market),
                      'Unavailable_target': int((~target_ok).sum()),
                      'Unavailable_features_with_target': int((target_ok & ~feature_ok).sum()),
                      'Eligible': len(part)})
    assert used == df.index.tolist()
    final = pd.concat(datasets.values(), ignore_index=True)
    assert final.Date.is_unique and final.Date.is_monotonic_increasing
    assert len(final) == sum(len(part) for part in datasets.values())
    for left, right in [('train', 'validation'), ('validation', 'test')]:
        assert datasets[left].Outcome_date.max() < datasets[right].Date.min()
    recent = df.tail(5)
    recent_features = X.loc[recent.index]
    prediction = recent[['Date']].join(recent_features).dropna().copy()
    assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(prediction.columns)
    assert set(prediction.Date).isdisjoint(final.Date)
    return final, datasets, prediction, pd.DataFrame(audit)

### 9.3 Validate and summarize the eligible observations
Record exclusions separately for unavailable outcomes and unavailable features. Retain the latest five sessions with complete predictors separately for inference; no future labels are attached.

In [ ]:
final_dataset, datasets, prediction_inputs, dataset_audit = build_datasets(df, X, partitions, make_targets)
print('Dataset preparation audit:')
display(dataset_audit)
print('Final research dataset:', final_dataset.shape)
print('Prediction inputs:', prediction_inputs.shape)

X_fit = datasets['train'][FEATURE_COLUMNS].copy()
y_fit = datasets['train']['Target'].copy()
assert X_fit.index.equals(y_fit.index)
assert len(X_fit) == len(training_labels.loc[training_mask])
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct', 'Partition'}.intersection(X_fit.columns)
print('Training input shape:', X_fit.shape)
print('Training target shape:', y_fit.shape)

### 9.4 Export reproducible datasets
Export the labelled research table, chronological subsets, inference inputs and an exclusion audit. Source snapshots are not overwritten. Validation and test outcomes are not summarized during preparation.

In [ ]:
export_dir = Path(os.environ.get('KAIROS_OUTPUT_DIR', str(base / 'kairos_datasets')))
export_dir.mkdir(parents=True, exist_ok=True)
exports = {'KAIROS_COMBINED_CLEANED.csv': combined,
           'KAIROS_VIX_IMPUTATION_AUDIT.csv': df.loc[df.VIX_missing.eq(1), ['Date', 'VIX_Close']],
           'KAIROS_FINAL_DATASET.csv': final_dataset,
           'KAIROS_TRAIN.csv': datasets['train'],
           'KAIROS_VALIDATION.csv': datasets['validation'],
           'KAIROS_TEST.csv': datasets['test'],
           'KAIROS_PREDICTION_INPUTS.csv': prediction_inputs,
           'KAIROS_DATASET_AUDIT.csv': dataset_audit}
for filename, frame in exports.items():
    path = export_dir / filename
    frame.to_csv(path, index=False)
    saved = pd.read_csv(path)
    assert saved.shape == frame.shape
    assert saved.columns.tolist() == frame.columns.tolist()
    print(filename, saved.shape)

## 10. Evaluation metrics and benchmark rules
Use the training majority class and trailing five-session momentum as simple comparison rules, not deployed models.

In [ ]:
"""Validation benchmarks for five-session market movement classification."""
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

CLASSES = [-1, 0, 1]
CLASS_NAMES = ['Major Down', 'Ordinary', 'Major Up']


def momentum_predict(X, threshold=3.0):
    """Classify trailing five-session return using the target thresholds."""
    if threshold <= 0:
        raise ValueError('Threshold must be positive')
    returns = X['Return_5d'].to_numpy(dtype=float)
    if not np.isfinite(returns).all():
        raise ValueError('Momentum inputs must be finite')
    boundary = np.isclose(np.abs(returns), threshold, rtol=0, atol=1e-10)
    return np.select([(returns < -threshold) & ~boundary,
                      (returns > threshold) & ~boundary], [-1, 1], default=0)


def evaluate_predictions(y_true, y_pred):
    """Score every target class, including classes never predicted."""
    report = classification_report(y_true, y_pred, labels=CLASSES,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    summary = {'Accuracy': accuracy_score(y_true, y_pred),
               'Balanced accuracy': balanced_accuracy_score(y_true, y_pred),
               'Macro F1': f1_score(y_true, y_pred, labels=CLASSES,
                                    average='macro', zero_division=0),
               'Major Down recall': report['Major Down']['recall'],
               'Major Up recall': report['Major Up']['recall']}
    per_class = pd.DataFrame(report).T.loc[CLASS_NAMES,
                                         ['precision', 'recall', 'f1-score', 'support']]
    matrix = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES),
                          index=CLASS_NAMES, columns=CLASS_NAMES)
    return summary, per_class, matrix

## 11. Single Random Forest
Use the previously locked V1 configuration, fitted only on the original training partition. No new search or test-driven tuning.

In [ ]:
"""The single KAIROS directional model and inference contract."""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

FEATURES = ['Return_1d','Return_5d','Volatility_20d','MA_20_gap_pct','Intraday_range_pct','VIX_Close','VIX_change_1d']


def build_reference():
    return RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10,
                                  criterion='gini', max_features='sqrt', bootstrap=True,
                                  class_weight='balanced', random_state=42, n_jobs=1)


def predict_reference(model, frame):
    if not set(['Date'] + FEATURES).issubset(frame.columns):
        raise ValueError('Missing required columns')
    dates = pd.to_datetime(frame.Date, errors='raise')
    if dates.isna().any() or dates.duplicated().any() or not dates.is_monotonic_increasing:
        raise ValueError('Dates must be unique and chronological')
    inputs = frame[FEATURES].astype(float)
    if not np.isfinite(inputs.to_numpy()).all():
        raise ValueError('Nonfinite inputs')
    assert model.classes_.tolist() == [-1, 0, 1]
    p = model.predict_proba(inputs)
    pred = model.predict(inputs)
    np.testing.assert_allclose(p.sum(axis=1), 1)
    np.testing.assert_array_equal(pred, model.classes_[p.argmax(axis=1)])
    return pd.DataFrame({'Date':dates, 'Prediction':pred, 'Score_Down':p[:,0],
                         'Score_Ordinary':p[:,1], 'Score_Up':p[:,2]}, index=frame.index)

## 12. Validation and historical test evaluation
Compare the one fitted model with two non-ML benchmark rules. The historical test period has already been examined; this is a reproducibility check, not independent validation.

**Future improvement target:** Major Down recall of at least 0.283 (28.3%) on a new chronological evaluation period. This is an aspirational benchmark, not an achieved historical result. The current model did not correctly classify any of the 23 historical test declines. On a set of 23 declines, meeting the target would require at least seven correct classifications.

In [ ]:
assert FEATURE_COLUMNS == FEATURES
assert datasets['train'].Outcome_date.max() < datasets['validation'].Date.min()
assert datasets['train'].Outcome_date.max() < datasets['test'].Date.min()
model = build_reference().fit(X_fit, y_fit)
assert model.n_features_in_ == 7
rows = []
for period in ('validation', 'test'):
    part = datasets[period]
    rec = predict_reference(model, part)
    rec['Outcome_date'] = part.Outcome_date
    rec['Forward_return_5d_pct'] = part.Forward_return_5d_pct
    rec['Target'] = part.Target
    rec['Correct'] = rec.Prediction.eq(rec.Target)
    for name, pred in [('Majority rule', np.full(len(part), y_fit.mode().iloc[0])),
                       ('Momentum rule', momentum_predict(part)), ('Random Forest', rec.Prediction)]:
        summary, report, matrix = evaluate_predictions(part.Target, pred)
        rows.append({'Period':period, 'Model':name, **summary})
        if name == 'Random Forest':
            print(period, 'Random Forest')
            display(report.round(4), matrix)
    rec.to_csv(export_dir / f'KAIROS_{period.upper()}_PREDICTIONS.csv', index=False)
    if period == 'test':
        records = rec
metrics = pd.DataFrame(rows)
display(metrics.round(4))
metrics.to_csv(export_dir / 'KAIROS_METRICS.csv', index=False)

## 13. Save the trained model and generate snapshot predictions
Reload the single model bundle and verify identical outputs. The latest five observations do not yet have known five-session outcomes in the snapshot.

In [ ]:
import joblib
import sklearn
import json
bundle = {'model':model, 'features':FEATURES, 'horizon':5, 'threshold_pct':3.0,
          'sklearn_version':sklearn.__version__, 'training_rows':len(X_fit),
          'training_last_outcome':str(datasets['train'].Outcome_date.max().date())}
joblib.dump(bundle, export_dir / 'KAIROS_REFERENCE.joblib', compress=3)
loaded = joblib.load(export_dir / 'KAIROS_REFERENCE.joblib')
pd.testing.assert_frame_equal(predict_reference(model, datasets['test']), predict_reference(loaded['model'], datasets['test']))
(export_dir / 'KAIROS_MODEL_MANIFEST.json').write_text(json.dumps({k:v for k,v in bundle.items() if k!='model'}, indent=2))
latest = predict_reference(model, prediction_inputs)
latest.to_csv(export_dir / 'KAIROS_LATEST_PREDICTIONS.csv', index=False)
display(latest.round(4))
print('Saved and verified one trained Random Forest.')

## 14. Explainable AI
Explain the locked Random Forest using impurity importance, validation permutation importance (20 repeats, macro F1), and path-dependent Tree SHAP for each class. SHAP values add to the uncalibrated class score; they do not establish causality. Test errors are retrospective diagnostics only. Install shap==0.50.0 if unavailable.

In [ ]:
"""Post-hoc explanations for the locked Random Forest; never fit or tune it."""
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, f1_score


def global_importance(model, X, y, repeats=20):
    scorer = make_scorer(f1_score, labels=[-1, 0, 1], average='macro', zero_division=0)
    p = permutation_importance(model, X, y, scoring=scorer, n_repeats=repeats,
                               random_state=42, n_jobs=1)
    return pd.DataFrame({'Feature': X.columns, 'Impurity importance': model.feature_importances_,
                         'Permutation mean': p.importances_mean,
                         'Permutation SD': p.importances_std}).sort_values('Permutation mean', ascending=False)


def explain_rows(model, frame, features):
    X = frame[features]
    explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent', model_output='raw')
    values = np.asarray(explainer.shap_values(X))
    base = np.asarray(explainer.expected_value)
    assert model.classes_.tolist() == [-1, 0, 1]
    assert values.shape == (len(X), len(features), 3) and base.shape == (3,)
    probabilities = model.predict_proba(X)
    np.testing.assert_allclose(base + values.sum(axis=1), probabilities, atol=1e-7, rtol=0)
    rows = []
    for k, label in enumerate(model.classes_):
        for j, feature in enumerate(features):
            rows.append(pd.DataFrame({'Date': frame.Date.to_numpy(), 'Class': label,
                'Feature': feature, 'Value': X[feature].to_numpy(), 'SHAP': values[:, j, k],
                'Base_score': base[k], 'Model_score': probabilities[:, k]}))
    return pd.concat(rows, ignore_index=True)


def error_summary(records):
    result = []
    for label in [-1, 0, 1]:
        subset = records.loc[records.Target.eq(label)]
        result.append({'Actual class': label, 'Rows': len(subset),
                       'Predicted down': int(subset.Prediction.eq(-1).sum()),
                       'Predicted ordinary': int(subset.Prediction.eq(0).sum()),
                       'Predicted up': int(subset.Prediction.eq(1).sum()),
                       'Correct': int(subset.Target.eq(subset.Prediction).sum())})
    return pd.DataFrame(result)

In [ ]:
xai_dir = export_dir / 'xai'
xai_dir.mkdir(parents=True, exist_ok=True)
importance = global_importance(model, datasets['validation'][FEATURES], datasets['validation'].Target)
display(importance.round(4))
importance.to_csv(xai_dir / 'KAIROS_FEATURE_IMPORTANCE.csv', index=False)
ax = importance.sort_values('Permutation mean').plot.barh(x='Feature', y='Permutation mean',
    xerr='Permutation SD', legend=False, figsize=(8, 4), title='Validation permutation importance')
ax.set_xlabel('Decrease in macro F1 (mean +/- permutation SD)')
ax.axvline(0, color='0.5', linewidth=1)
plt.tight_layout()
plt.show()

shap_test = explain_rows(model, datasets['test'], FEATURES)
shap_latest = explain_rows(model, prediction_inputs, FEATURES)
shap_test.to_csv(xai_dir / 'KAIROS_TEST_SHAP.csv', index=False)
shap_latest.to_csv(xai_dir / 'KAIROS_LATEST_SHAP.csv', index=False)
errors = error_summary(records)
display(errors)
errors.to_csv(xai_dir / 'KAIROS_ERROR_SUMMARY.csv', index=False)

cases = []
for name, mask in [('First missed decline', records.Target.eq(-1) & records.Prediction.ne(-1)),
                   ('First correct rise', records.Target.eq(1) & records.Prediction.eq(1)),
                   ('First correct ordinary', records.Target.eq(0) & records.Prediction.eq(0))]:
    if mask.any():
        row = records.loc[mask].iloc[0]
        cases.append({'Case':name, **row.to_dict()})
case_table = pd.DataFrame(cases)
case_table.to_csv(xai_dir / 'KAIROS_EXAMPLE_CASES.csv', index=False)
display(case_table)
for case in cases:
    local = shap_test.loc[shap_test.Date.eq(case['Date']) & shap_test.Class.eq(case['Prediction'])]
    display(local[['Feature','Value','SHAP','Base_score','Model_score']].round(4))
    ax = local.sort_values('SHAP').plot.barh(x='Feature', y='SHAP', legend=False,
        title=case['Case'] + ' - predicted class', figsize=(8, 4))
    ax.axvline(0, color='0.5', linewidth=1)
    ax.set_xlabel('Contribution to predicted class score')
    plt.tight_layout()
    plt.show()

## 15. Limitations
The target is a provisional five-session return threshold of ±3%. The historical test period has already been inspected and is not independent external validation. Overlapping targets are dependent. Scores are uncalibrated; the original model missed every major decline in the historical test period. No trading reliability is established. Earlier experimental branches remain in GitHub history. SHAP explains model scores, not causal effects; permutation importance is sensitive to correlated features and temporal dependence.

## 16. Local dashboard
The eight-tab Streamlit application is separate from this research notebook. Use the complete local package or place the two original CSVs under `data/` and copy the notebook export directory into `kairos_datasets/` at the repository root. All dashboard CSV exports are now generated directly in that output directory. Install the repository requirements, then run `python -m streamlit run dashboard/app.py` from the repository root. It uses exactly one saved Random Forest, without retraining, and provides historical predictions and SHAP explanations. The snapshot is not a live feed. All 19 weekend dates have supporting special-session evidence. Full official price/calendar reconciliation and browser testing remain pending.